# 46 — Resume vs JD Matching
**Goal:** Build a complete resume-to-job-description matcher using multiple signals.

## 1. Multi-Signal Matching Strategy

In [ ]:
print('''Resume-JD matching uses multiple signals:
1. Skill overlap (exact + semantic) -> 40% weight
2. Experience level match -> 20% weight
3. Education match -> 15% weight
4. Embedding similarity -> 25% weight

Final score = weighted combination of all signals.
Confidence comes from agreement between signals.''')

## 2. Building the Matcher

In [ ]:
from sentence_transformers import SentenceTransformer, util
import re
from rapidfuzz import fuzz

class ResumeJDMatcher:
    def __init__(self):
        self.skills_db = ["Python", "TensorFlow", "PyTorch", "NLP", "SQL", "AWS", "Docker", "Spark", "Java", "React"]
        try:
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.has_model = True
        except:
            self.has_model = False
    
    def skill_overlap(self, resume_skills, jd_skills):
        if not resume_skills or not jd_skills: return 0
        matched = sum(1 for s in resume_skills if any(
            fuzz.partial_ratio(s.lower(), j.lower()) > 85 for j in jd_skills
        ))
        return matched / max(len(jd_skills), 1)
    
    def embedding_match(self, resume_text, jd_text):
        if not self.has_model: return 0.5
        emb1 = self.model.encode(resume_text[:512])
        emb2 = self.model.encode(jd_text[:512])
        return util.cos_sim(emb1, emb2).item()
    
    def match(self, resume_text, jd_text):
        # Extract skills from both
        resume_skills = [s for s in self.skills_db if s.lower() in resume_text.lower()]
        jd_skills = [s for s in self.skills_db if s.lower() in jd_text.lower()]
        
        skill_score = self.skill_overlap(resume_skills, jd_skills) * 0.40
        emb_score = self.embedding_match(resume_text, jd_text) * 0.25
        # Experience and education bonuses (simplified)
        exp_bonus = 0.20 if re.search(r"\\d\\+?\\s*years?", resume_text, re.IGNORECASE) else 0.0
        edu_bonus = 0.15 if re.search(r"(masters|phd|bachelor)", resume_text, re.IGNORECASE) else 0.0
        
        total = skill_score + emb_score + exp_bonus + edu_bonus
        return {
            "score": round(total, 3),
            "details": {"skill_match": skill_score, "embedding": emb_score,
                        "experience": exp_bonus, "education": edu_bonus},
            "resume_skills_found": resume_skills,
            "jd_skills_required": jd_skills,
        }

matcher = ResumeJDMatcher()
resume = """Senior Data Scientist with 5+ years Python, NLP, TensorFlow experience.
Masters in Computer Science."""
jd = """Senior Data Scientist required. Python, TensorFlow, NLP preferred.
5+ years experience. MS/PhD preferred."""

result = matcher.match(resume, jd)
print(f"Match score: {result['score']}")
print(f"Breakdown: {result['details']}")
print(f"Resume skills: {result['resume_skills_found']}")
print(f"JD requires: {result['jd_skills_required']}")

## 3. Testing on Multiple JDs

In [ ]:
jds = [
    "Data Scientist — Python, ML, NLP, TensorFlow required",
    "Java Backend Developer — Spring, Microservices, AWS",
    "DevOps Engineer — Docker, Kubernetes, CI/CD",
    "Frontend Developer — React, TypeScript, CSS",
]
for i, jd_text in enumerate(jds):
    r = matcher.match(resume, jd_text)
    print(f"  JD {i+1}: {r['score']:.3f} — {jd_text}")

## Summary: Multi-signal matching with weighted scoring. Transparent, debuggable, no black box.